# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LaibaSabir1/flyrank-ml-internship-laiba_sabir/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/LaibaSabir1/flyrank-ml-internship-laiba_sabir"
REPO_DIR = "flyrank-ml-internship-laiba_sabir"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found.\n")

# Rebuild the pipeline artifacts this notebook depends on (feature vector, baseline, trained models).
# scripts/ stays untouched — I'm running it, not editing it (see GUIDE.md section 1).
for script in ["01_prepare_features.py", "02_baseline_score.py", "03_train_model.py"]:
    print(f"--- running scripts/{script} ---")
    subprocess.run([sys.executable, f"scripts/{script}"], check=True)

print("\nArtifacts ready: data/processed/*.csv and outputs/model_results.json")

Working dir: /content/flyrank-ml-internship-laiba_sabir
Starter data found.

--- running scripts/01_prepare_features.py ---
--- running scripts/02_baseline_score.py ---
--- running scripts/03_train_model.py ---

Artifacts ready: data/processed/*.csv and outputs/model_results.json


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [2]:
from IPython.display import Markdown, display

display(Markdown("""
**My method: Random Forest, checked against Logistic Regression and a Decision Tree.**

My lane is ranking, not classification — the real decision is "review these 50 pages first,"
not "is this one page declining, yes or no." That points at models whose *probabilities* can
rank pages, evaluated with Precision@K rather than plain accuracy.

I ran the ladder from simple to complex, on purpose:
- **Logistic Regression** — a linear, fully-readable starting point. If this alone beat the
  baseline by a lot, I wouldn't need anything fancier.
- **Decision Tree** (`max_depth=5`) — captures non-linear splits (e.g. "high impressions AND
  old content") that a linear model can't, and I can still print and read it.
- **Random Forest** (`n_estimators=200`, `max_depth=10`) — my main pick. Content-decline signals
  are tangled (impressions, position, age, freshness all interact), which is exactly where
  a single readable tree runs out of capacity but a forest of trees can still be explained
  through feature importance and permutation importance (Section 4).

I picked Random Forest as the **best model** using the same selection rule as the reference
pipeline: whichever model has the highest **Precision@50** on the held-out test set (ties
broken by average precision, then ROC AUC) — because Precision@50 is the metric that matches
my actual decision: a reviewer works through 50 pages a week, not the whole 30,000.

I did *not* reach for Gradient Boosting here. Random Forest already clears the baseline by a
wide margin (Section 3), and the extra tuning complexity of boosting isn't earning its keep yet
— per `training-honest-models`, complexity has to be justified by the comparison table, not
assumed.
"""))


**My method: Random Forest, checked against Logistic Regression and a Decision Tree.**

My lane is ranking, not classification — the real decision is "review these 50 pages first,"
not "is this one page declining, yes or no." That points at models whose *probabilities* can
rank pages, evaluated with Precision@K rather than plain accuracy.

I ran the ladder from simple to complex, on purpose:
- **Logistic Regression** — a linear, fully-readable starting point. If this alone beat the
  baseline by a lot, I wouldn't need anything fancier.
- **Decision Tree** (`max_depth=5`) — captures non-linear splits (e.g. "high impressions AND
  old content") that a linear model can't, and I can still print and read it.
- **Random Forest** (`n_estimators=200`, `max_depth=10`) — my main pick. Content-decline signals
  are tangled (impressions, position, age, freshness all interact), which is exactly where
  a single readable tree runs out of capacity but a forest of trees can still be explained
  through feature importance and permutation importance (Section 4).

I picked Random Forest as the **best model** using the same selection rule as the reference
pipeline: whichever model has the highest **Precision@50** on the held-out test set (ties
broken by average precision, then ROC AUC) — because Precision@50 is the metric that matches
my actual decision: a reviewer works through 50 pages a week, not the whole 30,000.

I did *not* reach for Gradient Boosting here. Random Forest already clears the baseline by a
wide margin (Section 3), and the extra tuning complexity of boosting isn't earning its keep yet
— per `training-honest-models`, complexity has to be justified by the comparison table, not
assumed.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [3]:
display(Markdown("""
**My split: client-holdout (grouped by `client_id`), not a random row split.**

This dataset has 32 pseudonymized clients, and pages from the same client likely share
hidden character — a client's site template, content strategy, or niche competitiveness
shows up across many of that client's pages at once. A random row split lets pages from the
*same* client sit in both train and test, so the model can partly memorize "this client's
pages behave like X" rather than learning a signal that generalizes to a **client it has
never seen** — which is the real deployment scenario (a new client's pages need ranking too).

`scripts/03_train_model.py` implements this as `client_holdout`: it shuffles the unique
`client_id`s, holds out ~20% of *clients* (not rows), and only falls back to a stratified row
split if there are too few clients or a class goes missing in either half. Both my baseline
and my model are evaluated on that same held-out set of clients.
"""))

import json
results = json.load(open("outputs/model_results.json"))
print("Split strategy used:", results["split_strategy"])
train_rows = results["train_rows"]
test_rows = results["test_rows"]
pos_rate = results["target_positive_rate"]
print(f"Train rows: {train_rows:,}   Test rows: {test_rows:,}")
print(f"Positive rate (whole dataset): {pos_rate:.3f}")


**My split: client-holdout (grouped by `client_id`), not a random row split.**

This dataset has 32 pseudonymized clients, and pages from the same client likely share
hidden character — a client's site template, content strategy, or niche competitiveness
shows up across many of that client's pages at once. A random row split lets pages from the
*same* client sit in both train and test, so the model can partly memorize "this client's
pages behave like X" rather than learning a signal that generalizes to a **client it has
never seen** — which is the real deployment scenario (a new client's pages need ranking too).

`scripts/03_train_model.py` implements this as `client_holdout`: it shuffles the unique
`client_id`s, holds out ~20% of *clients* (not rows), and only falls back to a stratified row
split if there are too few clients or a class goes missing in either half. Both my baseline
and my model are evaluated on that same held-out set of clients.


Split strategy used: client_holdout
Train rows: 27,675   Test rows: 2,325
Positive rate (whole dataset): 0.542


In [4]:
import sys
sys.path.insert(0, "scripts")
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

frame = pd.read_csv("data/processed/refresh_feature_vector.csv")
numeric = [c for c in MODEL_NUMERIC_FEATURES if c in frame.columns]
categorical = [c for c in MODEL_CATEGORICAL_FEATURES if c in frame.columns]
num = frame[numeric].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat = pd.get_dummies(frame[categorical].fillna("unknown").astype(str), prefix=categorical, dtype=float)
X_all = pd.concat([num.reset_index(drop=True), cat.reset_index(drop=True)], axis=1)
y_all = frame["is_declining_label"].astype(int)

Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
rf_naive = RandomForestClassifier(
    class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
    n_estimators=200, n_jobs=-1, random_state=42,
).fit(Xtr_r, ytr_r)
naive_p50 = precision_at_k(yte_r, rf_naive.predict_proba(Xte_r)[:, 1], 50)

honest_p50 = results["models"]["random_forest"]["precision_at_50"]

print(f"Naive RANDOM row split   Precision@50: {naive_p50:.3f}")
print(f"Honest client-holdout    Precision@50: {honest_p50:.3f}")
print(f"\nGap: {naive_p50 - honest_p50:+.3f} — the random split looks better, but that gap is")
print("partly the model recognizing clients it already trained on, not real ranking skill.")
print("The client-holdout number is the one I report as my result.")

Naive RANDOM row split   Precision@50: 0.900
Honest client-holdout    Precision@50: 0.740

Gap: +0.160 — the random split looks better, but that gap is
partly the model recognizing clients it already trained on, not real ranking skill.
The client-holdout number is the one I report as my result.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
# Same test set (client-holdout), same metric (Precision@50), for baseline + all 3 models.
base_rate = results["target_positive_rate"]

rows = []
rows.append({
    "method": "baseline_rules (Week 4)",
    "roc_auc": results["baseline"]["baseline_roc_auc"],
    "avg_precision": results["baseline"]["baseline_average_precision"],
    "precision_at_20": results["baseline"]["baseline_precision_at_20"],
    "precision_at_50": results["baseline"]["baseline_precision_at_50"],
    "recall": results["baseline"]["baseline_recall"],
})
for name, m in results["models"].items():
    rows.append({
        "method": name,
        "roc_auc": m["roc_auc"],
        "avg_precision": m["average_precision"],
        "precision_at_20": m["precision_at_20"],
        "precision_at_50": m["precision_at_50"],
        "recall": m["recall"],
    })

comparison = pd.DataFrame(rows).set_index("method").round(3)
comparison.insert(0, "base_rate_for_reference", round(base_rate, 3))
print(f"Base rate (majority-class / naive ranking floor): {base_rate:.3f}")
comparison

Base rate (majority-class / naive ranking floor): 0.542


,base_rate_for_reference,roc_auc,avg_precision,precision_at_20,precision_at_50,recall
method,,,,,,
baseline_rules (Week 4),0.542,0.627,0.468,0.15,0.24,0.189
decision_tree,0.542,0.742,0.575,0.45,0.58,0.716
logistic_regression,0.542,0.700,0.522,0.35,0.40,0.567
random_forest,0.542,0.750,0.618,0.65,0.74,0.744


In [6]:
best_name = results["best_model"]["name"]
best_p50 = results["models"][best_name]["precision_at_50"]
base_p50 = results["baseline"]["baseline_precision_at_50"]

display(Markdown(f"""
**Result:** on the same client-holdout test set, `{best_name}` reaches Precision@50 of
**{best_p50:.3f}** versus the baseline rule's **{base_p50:.3f}** — roughly a
**{best_p50 / base_p50:.1f}x** lift. Of the top 50 pages the model would send a reviewer to
this week, about {round(best_p50 * 50)} are genuinely declining, versus about
{round(base_p50 * 50)} for the hand-written rule.

Decision Tree and Logistic Regression both beat the baseline too, but by less — the ladder
(linear → tree → forest) each step earns real Precision@50 gain here, which is what justifies
using the more complex model instead of stopping at logistic regression.
"""))


**Result:** on the same client-holdout test set, `random_forest` reaches Precision@50 of
**0.740** versus the baseline rule's **0.240** — roughly a
**3.1x** lift. Of the top 50 pages the model would send a reviewer to
this week, about 37 are genuinely declining, versus about
12 for the hand-written rule.

Decision Tree and Logistic Regression both beat the baseline too, but by less — the ladder
(linear → tree → forest) each step earns real Precision@50 gain here, which is what justifies
using the more complex model instead of stopping at logistic regression.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
# Rebuild the exact client-holdout split used by scripts/03_train_model.py so error analysis
# lines up with the reported numbers.
RANDOM_STATE = 42
all_indices = np.arange(len(frame))
client_series = frame["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = client_series.isin(test_clients).to_numpy()
train_idx, test_idx = all_indices[~test_mask], all_indices[test_mask]

Xtr, Xte = X_all.iloc[train_idx], X_all.iloc[test_idx]
ytr, yte = y_all.iloc[train_idx], y_all.iloc[test_idx]

rf = RandomForestClassifier(
    class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
    n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE,
).fit(Xtr, ytr)
proba = rf.predict_proba(Xte)[:, 1]

print("Recomputed client-holdout Precision@50:",
      round(precision_at_k(yte, proba, 50), 3), "(should match Section 3)")

Recomputed client-holdout Precision@50: 0.74 (should match Section 3)


In [8]:
# What does the model lean on? Built-in importance vs permutation importance (menu item:
# permutation importance) — they should broadly agree, or the built-in number is suspect.
from sklearn.inspection import permutation_importance

built_in = pd.Series(rf.feature_importances_, index=X_all.columns).sort_values(ascending=False).head(10)

perm = permutation_importance(
    rf, Xte, yte, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1, scoring="roc_auc"
)
perm_imp = pd.Series(perm.importances_mean, index=X_all.columns).sort_values(ascending=False).head(10)

importance_compare = pd.DataFrame({
    "built_in_importance": built_in,
    "permutation_importance (roc_auc drop)": perm.importances_mean[[list(X_all.columns).index(f) for f in built_in.index]],
}).round(4)
importance_compare

,built_in_importance,permutation_importance (roc_auc drop)
days_with_impressions,0.1350,0.0565
log_impressions_90d,0.1294,0.0218
avg_position,0.1092,0.0068
content_age_days,0.0920,0.0012
char_count,0.0387,-0.0013
age_tier_365+,0.0368,0.0009
log_clicks_90d,0.0366,0.0062
word_count,0.0354,-0.0065
ctr,0.0352,0.0105
scroll_rate,0.0339,0.0065


In [9]:
display(Markdown("""
Both methods agree on the top signal: **`days_with_impressions`** and **`log_impressions_90d`**
dominate — visibility and how consistently a page gets found matters more than any single
content property. `avg_position`, `content_age_days`, and `ctr` follow. This matches the
Week-4 signal audit finding that visibility does more predictive work than staleness alone,
and it means the model is leaning on genuinely observable, pre-decision signals — not
something suspicious (no single feature towers over the rest the way a leaked column would).
"""))


Both methods agree on the top signal: **`days_with_impressions`** and **`log_impressions_90d`**
dominate — visibility and how consistently a page gets found matters more than any single
content property. `avg_position`, `content_age_days`, and `ctr` follow. This matches the
Week-4 signal audit finding that visibility does more predictive work than staleness alone,
and it means the model is leaning on genuinely observable, pre-decision signals — not
something suspicious (no single feature towers over the rest the way a leaked column would).


In [10]:
# Error analysis: look at the model's actual mistakes in its own top-50 queue.
test_frame = frame.iloc[test_idx].copy().reset_index(drop=True)
test_frame["model_probability"] = proba
top50 = test_frame.sort_values("model_probability", ascending=False).head(50)
false_positives = top50[top50["is_declining_label"] == 0]

print(f"False positives in the top 50: {len(false_positives)} of 50")
false_positives[[
    "impressions_90d", "avg_position", "ctr", "content_age_days", "days_with_impressions"
]].describe().round(2)

False positives in the top 50: 13 of 50


,impressions_90d,avg_position,ctr,content_age_days,days_with_impressions
count,13.00,13.00,13.00,13.00,13.00
mean,1791.77,20.36,0.07,138.85,65.08
std,1875.72,8.82,0.12,28.12,25.37
min,352.00,7.40,0.00,104.00,24.00
25%,761.00,13.30,0.00,112.00,58.00
50%,1076.00,21.80,0.00,134.00,75.00
75%,1647.00,25.10,0.11,175.00,85.00
max,6250.00,35.90,0.39,175.00,88.00


In [11]:
# The other side: declining pages the model was LEAST confident about (false-negative risk).
declining_test = test_frame[test_frame["is_declining_label"] == 1].sort_values("model_probability")
lowest_scored = declining_test.head(5)[[
    "impressions_90d", "avg_position", "ctr", "content_age_days",
    "days_with_impressions", "model_probability"
]]
lowest_scored

,impressions_90d,avg_position,ctr,content_age_days,days_with_impressions,model_probability
456,1,0.0,0.00,91,1,0.079867
306,3,0.0,0.00,308,2,0.082196
2094,3,0.7,0.00,104,2,0.149546
1727,3,0.3,33.33,300,3,0.152184
444,3,2.7,0.00,290,2,0.163987


In [12]:
display(Markdown(f"""
**Where the model is wrong, in plain words:**

- **False positives** ({len(false_positives)} of the top 50): pages the model flagged as
  declining that actually weren't. They're not random noise — they average
  ~{false_positives['impressions_90d'].mean():.0f} impressions and an average position around
  {false_positives['avg_position'].mean():.0f}, i.e. moderately visible, mid-position pages
  that *look* like the declining profile (visible + aging) but happened to hold steady. This
  is the honest cost of a probabilistic ranking model: it can't be perfect, and a human
  reviewer catching a handful of "false alarms" per week is a reasonable trade for catching
  the real ones.

- **False negatives** (lowest-scored declining pages): every one of them has **1–3 total
  impressions over 90 days**. The model is confidently *not* flagging these — and that's
  arguably correct behavior, not a failure: with that little traffic, "declining" is barely
  distinguishable from noise, which is exactly the impressions-floor caution from my Week-4
  signal audit (`w04_signal_audit.ipynb`). The model has effectively learned the same floor I
  found by hand.

**Bottom line:** the Random Forest beats the Week-4 baseline by roughly {best_p50 / base_p50:.1f}x
on Precision@50, on the same client-holdout split, using only pre-decision observable signals —
and its mistakes cluster in sensible places (mid-visibility false alarms, near-zero-traffic
false negatives) rather than anywhere that suggests leakage or a broken split.
"""))


**Where the model is wrong, in plain words:**

- **False positives** (13 of the top 50): pages the model flagged as
  declining that actually weren't. They're not random noise — they average
  ~1792 impressions and an average position around
  20, i.e. moderately visible, mid-position pages
  that *look* like the declining profile (visible + aging) but happened to hold steady. This
  is the honest cost of a probabilistic ranking model: it can't be perfect, and a human
  reviewer catching a handful of "false alarms" per week is a reasonable trade for catching
  the real ones.

- **False negatives** (lowest-scored declining pages): every one of them has **1–3 total
  impressions over 90 days**. The model is confidently *not* flagging these — and that's
  arguably correct behavior, not a failure: with that little traffic, "declining" is barely
  distinguishable from noise, which is exactly the impressions-floor caution from my Week-4
  signal audit (`w04_signal_audit.ipynb`). The model has effectively learned the same floor I
  found by hand.

**Bottom line:** the Random Forest beats the Week-4 baseline by roughly 3.1x
on Precision@50, on the same client-holdout split, using only pre-decision observable signals —
and its mistakes cluster in sensible places (mid-visibility false alarms, near-zero-traffic
false negatives) rather than anywhere that suggests leakage or a broken split.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.